---
# 🏭 Klassenarbeit – Lernfeld 3
## *Steuerungen analysieren und anpassen*

| | |
|---|---|
| **Thema** | Modernisierung einer Sortieranlage (SA-1) |
| **Dauer** | 60 Minuten |
| **Hilfsmittel** | Formelsammlung, Unterlagen (kein Internet) |

---

> 📋 **Hinweis für Schüler/innen:**  
> Speichern Sie dieses Notebook zunächst in Ihr persönliches Verzeichnis:  
> **File → Save Notebook As** → in Ihren eigenen Ordner speichern.  
> Arbeiten Sie ausschließlich in Ihrer eigenen Kopie.

---

## Szenario

In einer Produktionshalle werden Kunststoff-Werkstücke nach **Größe** (klein / mittel / groß)  
und **Farbe** (rot / blau) sortiert. Die bisherige Anlage wurde mit einer  
**zeitgesteuerten Bandsteuerung** betrieben – ohne Sensorik.

Das Wartungspersonal meldet: Werkstücke landen regelmäßig auf der falschen Rutsche.

Sie werden beauftragt, die Anlage auf eine **sensorbasierte SPS-Steuerung** umzustellen.  
Der **virtuelle Leitstand** zeigt Ihnen die Anlage in Echtzeit – nutzen Sie ihn zur  
Informationsbeschaffung und zur Kontrolle Ihrer Entscheidungen.

---
> ⚠️ **Hinweis:** Der Leitstand zeigt das **Verhalten** der Anlage.  
> Er gibt keine Lösungen vor – Ihre Begründungen werden bewertet.


In [ ]:
import random, math, hashlib, datetime
import ipywidgets as widgets
from IPython.display import display, HTML, IFrame

_alle_widgets  = []
_feedback_jobs = []
simulation_gestartet = False

def registriere(wlist, output, check_fn, hinweis_fn):
    ws = wlist if isinstance(wlist, list) else [wlist]
    _alle_widgets.extend(ws)
    _feedback_jobs.append((output, check_fn, hinweis_fn))

    for w in ws:
        try:
            w.observe(lambda change, w=w: log(
                f"Antwort geändert",
                f"{type(w).__name__}: {str(change.get('new',''))[:40]}"
            ) if change['name'] == 'value' else None, names='value')
        except Exception:
            pass

def sperre_alle():
    for w in _alle_widgets:
        try: w.disabled = True
        except: pass

def zeige_feedback():
    for out, chk, hint in _feedback_jobs:
        with out:
            out.clear_output(wait=True)
            ok = chk()
            display(HTML(feedback_html(ok, hint() if not ok else "")))

def feedback_html(ok, hinweis=""):
    if ok:
        return ("<div style='padding:8px 12px;margin:4px 0;background:#d4edda;"
                "border-left:4px solid #28a745;border-radius:4px;font-size:.9em;'>"
                "✅ <b>Korrekt.</b></div>")
    return ("<div style='padding:8px 12px;margin:4px 0;background:#f8d7da;"
            "border-left:4px solid #dc3545;border-radius:4px;font-size:.9em;'>"
            f"❌ <b>Nicht korrekt.</b> {hinweis}</div>")


p_name  = ""
p_klass = ""


_versuche = {}   

def versuche_verbraucht(aufgabe_id, max_v=3):
    """True wenn alle Versuche aufgebraucht."""
    return _versuche.get(aufgabe_id, 0) >= max_v

def versuche_erhoehen(aufgabe_id):
    _versuche[aufgabe_id] = _versuche.get(aufgabe_id, 0) + 1

def versuche_anzeige(aufgabe_id, max_v=3):
    v = _versuche.get(aufgabe_id, 0)
    rest = max_v - v
    farbe = "#ffc107" if rest == 1 else ("#dc3545" if rest == 0 else "#6c757d")
    return (f"<span style='font-size:.8em;color:{farbe};'>"
            f"Versuche: {v}/{max_v}</span>")

def h(s):
    return hashlib.sha256(str(s).strip().upper().encode()).hexdigest()[:16]

import datetime


_aktionslog = []

def log(aktion, details=""):
    """Jeden Schüler-Schritt mit Zeitstempel protokollieren."""
    eintrag = {
        "zeit": datetime.datetime.now().strftime("%H:%M:%S"),
        "aktion": aktion,
        "details": str(details)[:80]
    }
    _aktionslog.append(eintrag)

def log_anzeigen():
    """Vollständiges Protokoll als HTML-Tabelle ausgeben."""
    if not _aktionslog:
        display(HTML("<i>Keine Aktionen protokolliert.</i>"))
        return
    zeilen = "".join(
        f"<tr style='background:{'#f8f9fa' if j%2==0 else 'white'};'>"
        f"<td style='padding:3px 10px;font-family:monospace;color:#666;'>{e['zeit']}</td>"
        f"<td style='padding:3px 10px;'>{e['aktion']}</td>"
        f"<td style='padding:3px 10px;color:#555;font-size:.85em;'>{e['details']}</td>"
        f"</tr>"
        for j, e in enumerate(_aktionslog)
    )
    display(HTML(
        "<div style='margin-top:12px;'>"
        "<b>📋 Aktionsprotokoll</b> "
        f"<small style='color:#666;'>({len(_aktionslog)} Einträge)</small>"
        "<table style='border-collapse:collapse;width:100%;font-size:.88em;margin-top:6px;'>"
        "<tr style='background:#dee2e6;'>"
        "<th style='padding:4px 10px;text-align:left;'>Uhrzeit</th>"
        "<th style='padding:4px 10px;text-align:left;'>Aktion</th>"
        "<th style='padding:4px 10px;text-align:left;'>Details</th>"
        "</tr>"
        f"{zeilen}</table></div>"
    ))

# ── Lösungs-Hashes (SHA-256, 16 Zeichen) ───────────────────────────────────
_s = 0; _t = 1; v_loesung = 0.0  # werden durch start_pruefung gesetzt
_H_NOT     = h('NOT')
_H_QUIT    = h('QUIT')
_H_SET_DOM = h('SET_DOM')
_H_SPS     = h('SPS')
_H_E01     = h('E01')
_H_KORREKT = h('KORREKT')
_H_KAT0    = h('KAT0')
_H_1J      = h('1J')

import os
import json as _json

def exportiere_abgabe():
    import datetime as _dt
    ts   = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = p_name.replace(" ","_").replace("/","_")
    fn   = f"abgabe_{slug}_{p_klass}_{ts}.json"
    ft = {}
    for aid,lbl,ref in [
        ("2.1b","Gleichung A0.0","gleichung_21"),
        ("2.2c","SR-Erklaerung","erklaerung_22"),
        ("3.1","VPS/SPS Begruendung","begruendung_31"),
        ("3.3a","Gleichung W2","gleichung_w2"),
        ("3.3b","Gleichung W3","gleichung_w3"),
        ("4.1a","Fehlerbeschreibung","beob_41"),
        ("4.2a","NOT-HALT Begruendung","begruendung_42"),
        ("5.1b","Sichtpruefung","sichtpruef"),
        ("6.1","Funktionsbeschreibung","funk_beschr"),
        ("6.2a","Fehlerursache","ursache_txt"),
        ("6.2b","Leitstand-Nutzen","leitstand_nutzen"),
        ("6.2c","Massnahme","massnahme_txt"),
    ]:
        try:    ft[aid]={"l":lbl,"a":globals()[ref].value}
        except: ft[aid]={"l":lbl,"a":""}
    try: ft["2.3"]={"l":"v_min","a":v_min_input.value}
    except: pass
    au = {}
    for aid,lbl,ref in [
        ("2.1a","Logik","logik_w"),("2.2a","SR S","sr_s"),("2.2b","SR R","sr_r"),
        ("2.2d","SR Konflikt","sr_konflikt"),("3.1","Konzept","konzept_w"),
        ("4.1b","Fehlerbit","fehler_bit"),("4.1c","Korrektur","korrektur_41"),
        ("4.2a","Kontakt","oeffner_w"),("4.2b","Stopp","stopp_kat"),
        ("4.2c","Farbe","farbe_w"),("5.1a","S1","schritt_1"),
        ("5.1a","S2","schritt_2"),("5.1a","S3","schritt_3"),
        ("5.1c","Frist","pruef_frist"),("6.2","Erkannt","erkannt_wie"),
    ]:
        try: au[f"{aid}_{ref}"]={"l":lbl,"w":globals()[ref].value}
        except: pass
    try: au["1.1_eva"]={k:v.value for k,v in eva_ws.items()}
    except: pass
    try: au["1.1_adr"]={k:v.value for k,v in adr_ws.items()}
    except: pass
    try: au["1.2_kette"]=list(wk_auswahl)
    except: pass
    try: au["2.1c_wt"]={f"{k[0]}/{k[1]}":v.value for k,v in wt_eingaben.items()}
    except: pass
    try: au["3.2_sens"]={s:[d.value for d in ds] for s,ds in sensor_dds_32.items()}
    except: pass
    ak=[]
    for o,c,_ in _feedback_jobs:
        try: ak.append(c())
        except: ak.append(None)
    dat={"meta":{"name":p_name,"klasse":p_klass,"zeit":ts},
         "versuche":_versuche,"log":_aktionslog,
         "freitexte":ft,"auswahlen":au,"auto_korrekt":ak}
    d=os.path.join(os.path.expanduser("~"),"abgaben")
    os.makedirs(d,exist_ok=True)
    fp=os.path.join(d,fn)
    with open(fp,"w",encoding="utf-8") as fh:
        _json.dump(dat,fh,ensure_ascii=False,indent=2)
    return fp


In [ ]:
out_init   = widgets.Output()
name_w     = widgets.Text(description='Name:',   placeholder='Vorname Nachname',
                          layout={'width':'300px'}, style={'description_width':'initial'})
klasse_w   = widgets.Text(description='Klasse:', placeholder='z.B. ET23a',
                          layout={'width':'300px'}, style={'description_width':'initial'})
btn_start  = widgets.Button(description="Prüfung starten", button_style='success',
                            icon='lock', layout={'width':'200px'})

def start_pruefung(b):
    global p_name, p_klass
    p_name  = name_w.value.strip()
    p_klass = klasse_w.value.strip()
    if not p_name:
        with out_init:
            out_init.clear_output()
            display(HTML("<div style='color:red;'>⚠️ Bitte Namen eingeben.</div>"))
        return
    if not p_klass:
        with out_init:
            out_init.clear_output()
            display(HTML("<div style='color:red;'>⚠️ Bitte Klasse eingeben.</div>"))
        return
    random.seed(hash(p_name.lower()))
    global _s, _t, v_loesung
    _s = random.randint(180, 340)
    _t = random.randint(4, 9)
    v_loesung = round(_s / _t, 2)
    with out_23_params:
        out_23_params.clear_output()
        display(HTML(f"""
<div style='border:2px solid #003366;padding:14px;border-radius:8px;
            background:#f0f4f8;max-width:420px;'>
  <h4 style='margin-top:0;color:#003366;'>Individuelle Anlagenparameter – {p_name}</h4>
  <table style='border-collapse:collapse;width:100%'>
    <tr><td style='padding:5px 12px;'>Abstand B1 → Weiche W1</td>
        <td style='padding:5px 12px;'><b>s = {_s} cm</b></td></tr>
    <tr style='background:#e8eef5;'>
        <td style='padding:5px 12px;'>Maximale Durchlaufzeit</td>
        <td style='padding:5px 12px;'><b>t = {_t} s</b></td></tr>
  </table>
  <small style='color:#666;'>Formel: v = s / t &nbsp;|&nbsp; Einheit: cm/s</small>
</div>
"""))
    name_w.disabled = klasse_w.disabled = btn_start.disabled = True
    btn_start.description = "✅ Gestartet"
    log("Prüfung gestartet", f"Name: {p_name} | Klasse: {p_klass}")
    with out_init:
        out_init.clear_output()
        display(HTML(
            f"<div style='background:#d4edda;border-left:4px solid #28a745;"
            f"padding:12px;border-radius:4px;'>"
            f"✅ <b>Prüfungsumgebung initialisiert</b><br>"
            f"Name: <b>{p_name}</b> | Klasse: <b>{p_klass}</b><br>"
            f"<small>Individuelle Aufgabenwerte wurden generiert.</small></div>"
        ))

btn_start.on_click(start_pruefung)
display(widgets.VBox([
    widgets.HTML("<b>Identitätsfeststellung – bitte vor dem Start ausfüllen</b>"),
    name_w, klasse_w, btn_start, out_init
]))


## 🖥️ Virtueller Leitstand – Sortieranlage SA-1

> **Bedienung:** Zelle ausführen → Werkstücktyp wählen → **▶ ANLAUF** → **+ Werkstück aufgeben**  
> Beobachten Sie: Welche Sensoren schalten? Welche Weiche öffnet? Was zeigt die SPS an?  
> ⚠️ Der Leitstand **dient der Informationsbeschaffung** – er gibt keine Lösungen vor.


In [ ]:
leitstand_html = r'''
<div id="ls" style="font-family:'Segoe UI',sans-serif;background:#0d1117;color:#c9d1d9;
     border:2px solid #21262d;border-radius:12px;max-width:940px;
     box-shadow:0 0 40px rgba(0,100,255,.1);overflow:hidden;">
  <div style="background:linear-gradient(90deg,#0d1f3c,#0a2a50);padding:12px 20px;
              display:flex;align-items:center;gap:14px;border-bottom:1px solid #21262d;">
    <div id="led" style="width:12px;height:12px;border-radius:50%;background:#da3633;
         box-shadow:0 0 8px #da3633;transition:all .3s;"></div>
    <div>
      <div style="font-size:1.05em;font-weight:700;color:#58a6ff;letter-spacing:.05em;">
        LEITSTAND – SORTIERANLAGE SA-1</div>
      <div id="status_txt" style="font-size:.72em;color:#8b949e;font-family:monospace;">
        BETRIEBSZUSTAND: AUS | BAND: GESTOPPT</div>
    </div>
    <div style="margin-left:auto;font-family:monospace;font-size:.85em;color:#3fb950;"
         id="ws_cnt">WS gesamt: 0</div>
  </div>
  <div style="padding:16px 20px 8px;">
  <svg id="svg_ls" viewBox="0 0 880 270" xmlns="http://www.w3.org/2000/svg"
       style="width:100%;border-radius:8px;background:#161b22;">
    <!-- Band -->
    <rect x="55" y="128" width="710" height="28" rx="4" fill="#21262d" stroke="#30363d" stroke-width="1.5"/>
    <rect id="band_anim" x="55" y="131" width="710" height="10" rx="2" fill="#1f6feb" opacity="0.0" style="transition:opacity .4s;"/>
    <circle cx="68"  cy="142" r="14" fill="#30363d" stroke="#484f58" stroke-width="2"/>
    <circle cx="750" cy="142" r="14" fill="#30363d" stroke="#484f58" stroke-width="2"/>
    <!-- WS -->
    <rect id="ws_rect" x="75" y="112" width="36" height="24" rx="5"
          fill="#f78166" stroke="#ff7b72" stroke-width="1.5" opacity="0"/>
    <text id="ws_lbl" x="93" y="128" text-anchor="middle" font-size="8"
          fill="white" font-weight="bold" opacity="0"></text>
    <!-- B1 -->
    <line x1="200" y1="106" x2="200" y2="158" stroke="#388bfd" stroke-width="1.5" stroke-dasharray="4,3" opacity="0.4"/>
    <rect id="b1_box" x="185" y="96" width="30" height="16" rx="3" fill="#21262d" stroke="#388bfd" stroke-width="1.5"/>
    <text x="200" y="108" text-anchor="middle" font-size="8" fill="#388bfd" font-weight="bold">B1</text>
    <text x="200" y="173" text-anchor="middle" font-size="7" fill="#6e7681">Lichtschranke</text>
    <text x="200" y="183" text-anchor="middle" font-size="7" fill="#6e7681">E 0.0</text>
    <!-- B2 -->
    <line x1="390" y1="106" x2="390" y2="158" stroke="#3fb950" stroke-width="1.5" stroke-dasharray="4,3" opacity="0.4"/>
    <rect id="b2_box" x="375" y="96" width="30" height="16" rx="3" fill="#21262d" stroke="#3fb950" stroke-width="1.5"/>
    <text x="390" y="108" text-anchor="middle" font-size="8" fill="#3fb950" font-weight="bold">B2</text>
    <text x="390" y="173" text-anchor="middle" font-size="7" fill="#6e7681">Größensensor</text>
    <text x="390" y="183" text-anchor="middle" font-size="7" fill="#6e7681">E 0.1</text>
    <!-- B4 -->
    <line x1="475" y1="106" x2="475" y2="158" stroke="#ffa657" stroke-width="1.5" stroke-dasharray="4,3" opacity="0.4"/>
    <rect id="b4_box" x="460" y="96" width="30" height="16" rx="3" fill="#21262d" stroke="#ffa657" stroke-width="1.5"/>
    <text x="475" y="108" text-anchor="middle" font-size="8" fill="#ffa657" font-weight="bold">B4</text>
    <text x="475" y="173" text-anchor="middle" font-size="7" fill="#6e7681">Gr&#246;&#223;ens. 2</text>
    <text x="475" y="183" text-anchor="middle" font-size="7" fill="#6e7681">E 0.3</text>
    <!-- B3 -->
    <line x1="560" y1="106" x2="560" y2="158" stroke="#d2a8ff" stroke-width="1.5" stroke-dasharray="4,3" opacity="0.4"/>
    <rect id="b3_box" x="545" y="96" width="30" height="16" rx="3" fill="#21262d" stroke="#d2a8ff" stroke-width="1.5"/>
    <text x="560" y="108" text-anchor="middle" font-size="8" fill="#d2a8ff" font-weight="bold">B3</text>
    <text x="560" y="173" text-anchor="middle" font-size="7" fill="#6e7681">Farbsensor</text>
    <text x="560" y="183" text-anchor="middle" font-size="7" fill="#6e7681">E 0.2</text>
    <!-- Weichen -->
    <polygon id="w1_poly" points="630,156 670,156 660,205 620,205"
             fill="#e3b341" opacity="0.25" stroke="#d29922" stroke-width="1.5"/>
    <text x="645" y="186" text-anchor="middle" font-size="9" fill="#e3b341" font-weight="bold">W1</text>
    <text x="645" y="220" text-anchor="middle" font-size="7" fill="#6e7681">klein · A0.1</text>
    <polygon id="w2_poly" points="675,156 715,156 705,205 665,205"
             fill="#79c0ff" opacity="0.25" stroke="#58a6ff" stroke-width="1.5"/>
    <text x="690" y="186" text-anchor="middle" font-size="9" fill="#79c0ff" font-weight="bold">W2</text>
    <text x="690" y="220" text-anchor="middle" font-size="7" fill="#6e7681">mittel · A0.2</text>
    <polygon id="w3_poly" points="720,156 760,156 750,205 710,205"
             fill="#56d364" opacity="0.25" stroke="#3fb950" stroke-width="1.5"/>
    <text x="735" y="186" text-anchor="middle" font-size="9" fill="#56d364" font-weight="bold">W3</text>
    <text x="735" y="220" text-anchor="middle" font-size="7" fill="#6e7681">groß · A0.3</text>
    <!-- NOT-HALT -->
    <rect id="not_halt_btn" x="14" y="196" width="52" height="32" rx="4"
          fill="#da3633" stroke="#f85149" stroke-width="2" style="cursor:pointer"/>
    <text x="40" y="209" text-anchor="middle" font-size="8" fill="white" font-weight="bold">NOT-</text>
    <text x="40" y="221" text-anchor="middle" font-size="8" fill="white" font-weight="bold">HALT</text>
    <!-- SPS -->
    <rect x="14" y="14" width="140" height="72" rx="6" fill="#161b22" stroke="#30363d" stroke-width="1.5"/>
    <text x="84" y="30" text-anchor="middle" font-size="8.5" fill="#58a6ff" font-weight="bold">SPS – Steuereinheit</text>
    <text id="e_disp" x="84" y="46" text-anchor="middle" font-size="7.5" fill="#6e7681" font-family="monospace">E: 0 0 0  0 0 0</text>
    <text id="a_disp" x="84" y="59" text-anchor="middle" font-size="7.5" fill="#6e7681" font-family="monospace">A: 0 0 0  0 0</text>
    <text id="prog_lbl" x="84" y="74" text-anchor="middle" font-size="7" fill="#3fb950">Programm: BEREIT</text>
    <!-- Info-Box -->
    <rect x="762" y="14" width="104" height="52" rx="4" fill="#161b22" stroke="#30363d" stroke-width="1"/>
    <text x="814" y="30" text-anchor="middle" font-size="7" fill="#6e7681">Letztes Werkstück</text>
    <text id="last_ws" x="814" y="54" text-anchor="middle" font-size="13"
          fill="#c9d1d9" font-family="monospace" font-weight="bold">–</text>
    <rect x="762" y="72" width="104" height="42" rx="4" fill="#161b22" stroke="#30363d" stroke-width="1"/>
    <text x="814" y="88" text-anchor="middle" font-size="7" fill="#6e7681">Fehlercode</text>
    <text id="err_disp" x="814" y="107" text-anchor="middle" font-size="11"
          fill="#3fb950" font-family="monospace">OK</text>
  </svg>
  </div>
  <div style="padding:4px 20px 14px;display:flex;gap:8px;flex-wrap:wrap;align-items:center;">
    <button onclick="startBand()" style="background:#238636;color:#fff;border:none;
      border-radius:6px;padding:7px 18px;cursor:pointer;font-weight:600;font-size:.88em;">
      ▶ ANLAUF</button>
    <button onclick="stopBand()" style="background:#da3633;color:#fff;border:none;
      border-radius:6px;padding:7px 18px;cursor:pointer;font-weight:600;font-size:.88em;">
      ■ STOP</button>
    <select id="ws_typ" style="background:#21262d;color:#c9d1d9;border:1px solid #30363d;
      border-radius:6px;padding:5px 10px;font-size:.83em;">
      <option value="klein_rot">Klein · Rot</option>
      <option value="klein_blau">Klein · Blau</option>
      <option value="mittel_rot">Mittel · Rot</option>
      <option value="mittel_blau">Mittel · Blau</option>
      <option value="gross_rot">Groß · Rot</option>
      <option value="gross_blau">Groß · Blau</option>
    </select>
    <button onclick="sendeWS()" style="background:#1f6feb;color:#fff;border:none;
      border-radius:6px;padding:7px 14px;cursor:pointer;font-size:.83em;">
      + Werkstück aufgeben</button>
  </div>
</div>
<script>
var bandAn=false,wsAktiv=false,wsX=75,wsGesamt=0,animId=null,notHalt=false,wsB4=0;
var wsTypAkt='klein_rot';
var wsFarben={'klein_rot':'#f78166','klein_blau':'#79c0ff',
  'mittel_rot':'#ffa657','mittel_blau':'#d2a8ff',
  'gross_rot':'#ff7b72','gross_blau':'#56d364'};
var wsW={'klein':30,'mittel':44,'gross':60};

function upSPS(e,a){
  document.getElementById('e_disp').textContent='E: '+e.slice(0,3).join(' ')+'  '+e.slice(3).join(' ');
  document.getElementById('a_disp').textContent='A: '+a.slice(0,3).join(' ')+'  '+a.slice(3).join(' ');
}
function startBand(){
  if(notHalt)return;
  bandAn=true;
  document.getElementById('led').style.cssText='width:12px;height:12px;border-radius:50%;background:#3fb950;box-shadow:0 0 10px #3fb950;transition:all .3s;';
  document.getElementById('band_anim').style.opacity='0.5';
  document.getElementById('status_txt').textContent='BETRIEBSZUSTAND: EIN | BAND: LÄUFT | A0.0=1';
  document.getElementById('prog_lbl').textContent='Programm: AKTIV';
  upSPS([0,0,0,0,0,0],[1,0,0,0,0]);
  if(!animId)animId=setInterval(animStep,35);
}
function stopBand(){
  bandAn=false;
  document.getElementById('led').style.cssText='width:12px;height:12px;border-radius:50%;background:#da3633;box-shadow:0 0 8px #da3633;transition:all .3s;';
  document.getElementById('band_anim').style.opacity='0.0';
  document.getElementById('status_txt').textContent='BETRIEBSZUSTAND: AUS | BAND: GESTOPPT';
  document.getElementById('prog_lbl').textContent='Programm: BEREIT';
  upSPS([0,0,0,0,0,0],[0,0,0,0,0]);
}
function sendeWS(){
  if(!bandAn||wsAktiv)return;
  wsTypAkt=document.getElementById('ws_typ').value;
  var t=wsTypAkt.split('_'),gr=t[0];
  var w=document.getElementById('ws_rect'),l=document.getElementById('ws_lbl');
  var br=wsW[gr];
  w.setAttribute('width',br);
  w.setAttribute('fill',wsFarben[wsTypAkt]);
  l.textContent=gr.charAt(0).toUpperCase()+'/'+t[1].charAt(0).toUpperCase();
  w.setAttribute('x',75); l.setAttribute('x',75+br/2);
  w.style.opacity='0.92'; l.style.opacity='1';
  wsX=75; wsAktiv=true; wsGesamt++; wsB4=0;
  document.getElementById('ws_cnt').textContent='WS gesamt: '+wsGesamt;
  document.getElementById('err_disp').textContent='OK';
  document.getElementById('err_disp').style.color='#3fb950';
}
function blinkSensor(id,col){
  var el=document.getElementById(id);
  el.setAttribute('fill',col);
  el.style.filter='drop-shadow(0 0 7px '+col+')';
}
function dimSensor(id){
  var el=document.getElementById(id);
  el.setAttribute('fill','#21262d');
  el.style.filter='none';
}
function blinkWeiche(id,col){
  var el=document.getElementById(id);
  el.style.opacity='0.95';
  el.style.filter='drop-shadow(0 0 10px '+col+')';
  setTimeout(function(){el.style.opacity='0.25';el.style.filter='none';},900);
}
function wsWeg(weiche,label,col,ausgaenge){
  blinkWeiche(weiche,col);
  document.getElementById('last_ws').textContent=label;
  wsAktiv=false;
  document.getElementById('ws_rect').style.opacity='0';
  document.getElementById('ws_lbl').style.opacity='0';
  upSPS([0,0,0,0,0,0],ausgaenge);
}
function animStep(){
  if(!bandAn||!wsAktiv)return;
  wsX+=3;
  var t=wsTypAkt.split('_'),gr=t[0],fa=t[1];
  var w=document.getElementById('ws_rect'),l=document.getElementById('ws_lbl');
  var br=wsW[gr];
  w.setAttribute('x',wsX); l.setAttribute('x',wsX+br/2);
  // B1
  if(wsX>188&&wsX<222){blinkSensor('b1_box','#388bfd');upSPS([1,0,0,0,0,0],[1,0,0,0,0]);}
  else dimSensor('b1_box');
  // B2
  if(wsX>378&&wsX<415){
    blinkSensor('b2_box','#3fb950');
    var g=(gr==='gross'||gr==='mittel')?1:0;
    upSPS([1,g,0,0,0,0],[1,0,0,0,0]);
  } else dimSensor('b2_box');
  // B4
  if(wsX>463&&wsX<500){
    blinkSensor('b4_box','#ffa657');
    wsB4=(gr==='gross')?1:0;
    var g4=(gr==='gross'||gr==='mittel')?1:0;
    upSPS([1,g4,0,wsB4,0,0],[1,0,0,0,0]);
  } else { if(wsX<463){wsB4=0;} dimSensor('b4_box'); }
  // B3
  if(wsX>548&&wsX<584){
    blinkSensor('b3_box','#d2a8ff');
    var g3=(gr==='gross'||gr==='mittel')?1:0;
    upSPS([1,g3,(fa==='rot')?1:0,wsB4,0,0],[1,0,0,0,0]);
  } else dimSensor('b3_box');
  // Weichen
  if(wsX>622&&wsX<668&&gr==='klein'){
    wsWeg('w1_poly','↓ W1 klein','#e3b341',[1,1,0,0,0]); return;
  }
  if(wsX>672&&wsX<718&&gr==='mittel'&&fa==='rot'){
    wsWeg('w2_poly','↓ W2 m-rot','#79c0ff',[1,0,1,0,0]); return;
  }
  if(wsX>722&&wsX<762&&(gr==='gross'||(gr==='mittel'&&fa==='blau'))){
    wsWeg('w3_poly','↓ W3 gr/m-blau','#56d364',[1,0,0,1,0]); return;
  }
  if(wsX>770){
    wsAktiv=false;
    w.style.opacity='0'; l.style.opacity='0';
    document.getElementById('err_disp').textContent='E-01';
    document.getElementById('err_disp').style.color='#f85149';
    document.getElementById('last_ws').textContent='⚠ E-01';
    upSPS([0,0,0,0,0,0],[0,0,0,0,1]);
  }
}
document.getElementById('not_halt_btn').onclick=function(){
  notHalt=!notHalt;
  if(notHalt){
    stopBand();
    this.style.cssText='x:14;y:196;width:52px;height:32px;cursor:pointer;fill:#f85149;stroke:#ff7b72;stroke-width:3;filter:drop-shadow(0 0 12px #f85149);border-radius:4px;';
    document.getElementById('status_txt').textContent='⚠ NOT-HALT AKTIV – Quittierung erforderlich!';
    document.getElementById('err_disp').textContent='E-00';
    document.getElementById('err_disp').style.color='#f85149';
  } else {
    this.style.cssText='cursor:pointer;';
    document.getElementById('status_txt').textContent='BETRIEBSZUSTAND: AUS | QUITTIERT';
    document.getElementById('err_disp').textContent='OK';
    document.getElementById('err_disp').style.color='#3fb950';
  }
};
</script>
'''
display(HTML(leitstand_html))


---
# Handlungsschritt 1: Informieren
### Systemanalyse am virtuellen Leitstand

Beobachten Sie die Anlage im Leitstand. Testen Sie **alle 6 Werkstücktypen** und  
notieren Sie das Verhalten der Sensoren und Weichen.

---


## Aufgabe 1.1: EVA-Analyse *(7 Punkte)*

Betrachten Sie die Sortieranlage SA-1 am Leitstand.  
Ordnen Sie die Komponenten dem **EVA-Prinzip** zu und nennen Sie  
jeweils die **SPS-Adresse** (z.B. E0.0 oder A0.1).


In [ ]:
# ── Aufgabe 1.1: EVA-Zuordnung ───────────────────────────────────
eva_opt = [("Bitte wählen...", None), ("Eingabe (E)", "E"),
           ("Verarbeitung (V)", "V"), ("Ausgabe (A)", "A")]

komponenten = ["Lichtschranke B1", "Größensensor B2", "Größensensor 2 B4",
               "Farbsensor B3", "Förderband-Motor M1", "Weiche W1 (klein)", "SPS-Steuereinheit"]
loesung_eva = {"Lichtschranke B1":"E", "Größensensor B2":"E", "Größensensor 2 B4":"E",
               "Farbsensor B3":"E",    "Förderband-Motor M1":"A",
               "Weiche W1 (klein)":"A","SPS-Steuereinheit":"V"}

eva_ws = {}
rows_eva = []
for k in komponenten:
    dd = widgets.Dropdown(options=eva_opt, description=f'{k}:',
                          style={'description_width':'200px'},
                          layout={'width':'420px'})
    eva_ws[k] = dd
    rows_eva.append(dd)

adr_ws = {}
rows_adr = []
adr_loesung = {"Lichtschranke B1":"E0.0","Größensensor B2":"E0.1",
               "Größensensor 2 B4":"E0.3","Farbsensor B3":"E0.2",
               "Förderband-Motor M1":"A0.0","Weiche W1 (klein)":"A0.1",
               "SPS-Steuereinheit":"–"}
for k in komponenten:
    t = widgets.Text(placeholder='z.B. E0.0', description='Adresse:',
                     layout={'width':'200px'},
                     style={'description_width':'60px'})
    adr_ws[k] = t
    rows_adr.append(t)

out_11 = widgets.Output()
registriere(
    list(eva_ws.values()) + list(adr_ws.values()), out_11,
    check_fn   = lambda: all(eva_ws[k].value == loesung_eva[k] for k in komponenten),
    hinweis_fn = lambda: "Überprüfen Sie: " + ", ".join(
        k for k in komponenten if eva_ws[k].value != loesung_eva[k]) + "."
)

rows_combined = [widgets.HBox([rows_eva[i], rows_adr[i]]) for i in range(len(komponenten))]
display(widgets.VBox([
    widgets.Label("Ordnen Sie EVA-Bereich und SPS-Adresse zu:"),
    *rows_combined, out_11
]))


## Aufgabe 1.2: Wirkungskette *(8 Punkte)*

Beschreiben Sie den **Signalfluss** vom Moment des Werkstück-Aufgebens  
bis zur Weichenöffnung in der richtigen Reihenfolge.


In [ ]:
wk_korrekt = [
    "Werkstück unterbricht Lichtschranke B1 -> E0.0 = 1",
    "B2 erkennt mittelgroße und große Werkstücke -> E0.1 = 1 oder 0",
    "B4 erkennt ausschließlich große Werkstücke -> E0.3 = 1 (groß) oder 0",
    "B3 erkennt Werkstückfarbe -> E0.2 = 1 (rot) oder 0 (blau)",
    "SPS verknüpft E0.1, E0.3 und E0.2 -> Ausgangsbit A0.x wird gesetzt",
    "Magnetventil öffnet zugewiesene Weiche -> Werkstück umgeleitet",
]

import random as _rr
_rnd = _rr.Random(7)
shuffled_wk = wk_korrekt.copy()
_rnd.shuffle(shuffled_wk)

wk_auswahl = []
out_wk = widgets.Output()

def add_wk(b):
    if b.description not in wk_auswahl:
        wk_auswahl.append(b.description)
        b.button_style = 'success'
        b.disabled = True
        with out_wk:
            out_wk.clear_output()
            zeilen = "".join(
                f"<div style='padding:5px 10px;margin:3px 0;"
                f"background:#1f2937;border-left:3px solid #3fb950;"
                f"border-radius:4px;font-size:.88em;"
                f"color:#e5e7eb;'>"
                f"[{i+1}] {s}</div>"
                for i, s in enumerate(wk_auswahl)
            )
            display(HTML(
                "<div style='margin-top:8px;padding:8px;"
                "background:#111827;border-radius:6px;'>"
                + zeilen + "</div>"
            ))

def reset_wk(b):
    wk_auswahl.clear()
    out_wk.clear_output()
    for btn in wk_buttons:
        btn.disabled = False
        btn.button_style = ''

wk_buttons = [
    widgets.Button(
        description=s,
        layout={'width':'520px', 'margin':'2px'},
        style={'font_size': '13px'}
    )
    for s in shuffled_wk
]
for b in wk_buttons:
    b.on_click(add_wk)
_alle_widgets.extend(wk_buttons)

reset_btn_wk = widgets.Button(
    description="Zurücksetzen",
    button_style='warning',
    layout={'width':'160px'}
)
reset_btn_wk.on_click(reset_wk)
_alle_widgets.append(reset_btn_wk)

registriere([], out_wk,
    check_fn   = lambda: wk_auswahl == wk_korrekt,
    hinweis_fn = lambda: (
        "1 Schritt falsch platziert → 4 von 8 Punkten."
        if len(wk_auswahl) == len(wk_korrekt) and
           sum(1 for a,c in zip(wk_auswahl, wk_korrekt) if a != c) == 1
        else "Mehrere Schritte falsch → 0 Punkte. Beginnen Sie beim ersten physischen Ereignis."
    )
)
display(widgets.VBox([
    widgets.Label("Klicken Sie die Schritte in der richtigen Reihenfolge:"),
    *wk_buttons,
    reset_btn_wk,
    out_wk
]))


---
# Handlungsschritt 2: Planen
### Logik & Signalverarbeitung

---


## Aufgabe 2.1: Logische Grundverknüpfungen *(12 Punkte)*

Das Förderband darf nur anlaufen wenn **alle drei** Bedingungen gleichzeitig erfüllt sind:

- Hauptschalter S1 geschlossen **(E0.3 = 1)**
- Schutztür geschlossen, Türkontakt B5 **(E0.4 = 1)**  
- Kein NOT-HALT aktiv, d.h. NOT-HALT-Kreis geschlossen **(E0.5 = 1)**

**a)** Wählen Sie die korrekte Grundverknüpfung für den Bandanlauf.  
**b)** Geben Sie die Boolesche Gleichung an: `A0.0 = ?`  
**c)** Stellen Sie die Wahrheitstafel für E0.3 und E0.4 auf (E0.5 = 1 vorausgesetzt).


In [ ]:
logik_w = widgets.RadioButtons(
    value=None,
    options=[
        ('UND (AND): Alle Eingänge müssen gleichzeitig aktiv sein', 'AND'),
        ('ODER (OR): Mindestens ein Eingang muss aktiv sein',        'OR'),
        ('NICHT (NOT): Eingang wird invertiert',                     'NOT'),
        ('XOR: Genau ein Eingang darf aktiv sein',                   'XOR'),
    ],
    description='',
    layout={'width':'600px'}
)
gleichung_21 = widgets.Text(
    placeholder='z.B. A0.0 = E0.3 * E0.4 * E0.5',
    description='b) Gleichung:',
    style={'description_width':'120px'},
    layout={'width':'460px'}
)

display(widgets.HTML(
    "<div style='margin-top:12px;'><b>c) Wahrheitstafel ausfüllen</b> "
    "(E0.5 = 1 vorausgesetzt):</div>"
))

wt_eingaben = {}
wt_rows_display = []
wt_loesung_21 = {}

wt_header = widgets.HTML(
    "<div style='display:flex;gap:4px;margin:6px 0;font-weight:bold;"
    "font-size:.88em;padding:4px 0;border-bottom:2px solid #dee2e6;'>"
    "<div style='width:100px;text-align:center;'>S1 (E0.3)</div>"
    "<div style='width:120px;text-align:center;'>Tür B5 (E0.4)</div>"
    "<div style='width:140px;text-align:center;'>Band A0.0</div>"
    "<div style='width:120px;text-align:center;'>Status</div>"
    "</div>"
)
wt_rows_display.append(wt_header)

for x1 in [0, 1]:
    for x2 in [0, 1]:
        y_korrekt = int(x1 == 1 and x2 == 1)
        wt_loesung_21[(x1, x2)] = y_korrekt
        dd = widgets.Dropdown(
            options=[('?', None), ('0', 0), ('1', 1)],
            layout={'width':'100px'}
        )
        wt_eingaben[(x1, x2)] = dd
        _alle_widgets.append(dd)
        bg = '#f8f9fa'
        row = widgets.HBox([
            widgets.HTML(
                f"<div style='width:100px;text-align:center;"
                f"padding:6px;background:{bg};'>{x1}</div>"
            ),
            widgets.HTML(
                f"<div style='width:120px;text-align:center;"
                f"padding:6px;background:{bg};'>{x2}</div>"
            ),
            dd,
            widgets.HTML(
                f"<div style='width:120px;padding:6px;"
                f"background:{bg};color:#6c757d;font-size:.85em;'>"
                f"E0.5=1</div>"
            ),
        ])
        wt_rows_display.append(row)

out_21 = widgets.Output()

def check_wt_21():
    for (x1, x2), dd in wt_eingaben.items():
        if dd.value != wt_loesung_21[(x1, x2)]:
            return False
    return True

def check_21_gesamt():
    logik_ok = logik_w.value == 'AND'
    try:
        wt_ok = all(
            wt_eingaben[(x1,x2)].value == wt_loesung_21[(x1,x2)]
            for x1 in [0,1] for x2 in [0,1]
        )
    except Exception:
        wt_ok = False
    return logik_ok and wt_ok

registriere(
    [logik_w, gleichung_21] + list(wt_eingaben.values()), out_21,
    check_fn   = check_21_gesamt,
    hinweis_fn = lambda: "Alle drei Sicherheitsbedingungen müssen gleichzeitig erfüllt sein – UND-Verknüpfung."
)

display(widgets.VBox([
    widgets.HTML("<b>a) Verknüpfung:</b>"),
    logik_w,
    widgets.HTML("<b>b) Boolesche Gleichung:</b>"),
    gleichung_21,
    *wt_rows_display,
    out_21
]))


## Aufgabe 2.2: Speicherfunktion – SR-Flipflop *(10 Punkte)*

Der NOT-HALT soll das Förderband sofort stoppen und den Zustand **speichern**  
bis eine bewusste Quittierung erfolgt. Dafür wird ein **SR-Flipflop** eingesetzt.

**a)** Welches Signal übernimmt **S (Set)** – also das Setzen des Stoppzustands?  
**b)** Welches Signal übernimmt **R (Reset)** – also das Quittieren?  
**c)** Erklären Sie den Unterschied zwischen einer speichernden und nicht-speichernden Steuerung.  
**d)** Beim SR-Flipflop gilt: S ist dominant. Was bedeutet das für S = R = 1?


In [ ]:
sr_s = widgets.Dropdown(
    options=[
        ("Bitte wählen...", None),
        ("NOT-HALT-Taster (E0.5)", "NOT"),
        ("Start-Taster S1 (E0.3)", "START"),
        ("Quittier-Taster (E0.6)", "QUIT"),
    ],
    description='a) S (Set):',
    style={'description_width':'120px'},
    layout={'width':'360px'}
)
sr_r = widgets.Dropdown(
    options=[
        ("Bitte wählen...", None),
        ("Quittier-Taster (E0.6)", "QUIT"),
        ("NOT-HALT-Taster (E0.5)", "NOT"),
        ("Lichtschranke B1 (E0.0)", "LICHT"),
    ],
    description='b) R (Reset):',
    style={'description_width':'120px'},
    layout={'width':'360px'}
)
erklaerung_22 = widgets.Textarea(
    placeholder='c) Nicht-speichernd: Wenn der NOT-HALT-Taster losgelassen wird...'
                ' Speichernd (SR-Flipflop): Der Zustand bleibt erhalten bis...',
    description='c) Erklärung:',
    style={'description_width':'120px'},
    layout={'width':'90%', 'height':'80px'}
)
sr_konflikt = widgets.RadioButtons(
    value=None,
    options=[
        ("Q = 1, weil beim SR-Flipflop S (Set) dominant ist", "SET_DOM"),
        ("Q = 0, weil Reset immer Vorrang hat",                "RST_DOM"),
        ("Undefinierter Zustand – nicht vorhersagbar",         "UNDEF"),
    ],
    description='',
    layout={'width':'560px'}
)
out_22 = widgets.Output()

registriere(
    [sr_s, sr_r, sr_konflikt, erklaerung_22], out_22,
    check_fn   = lambda: (h(sr_s.value or '') == _H_NOT
                          and h(sr_r.value or '') == _H_QUIT
                          and h(sr_konflikt.value or '') == _H_SET_DOM),
    hinweis_fn = lambda: " ".join(filter(None, [
        "a) S setzt den Stoppzustand – das ist der NOT-HALT-Taster." if sr_s.value != 'NOT' else "",
        "b) R hebt den Zustand auf – das ist die Quittierung." if sr_r.value != 'QUIT' else "",
        "d) Beim SR-Flipflop ist S dominant: S=R=1 -> Q=1." if sr_konflikt.value != 'SET_DOM' else "",
    ]))
)
display(widgets.VBox([
    sr_s, sr_r, erklaerung_22,
    widgets.HTML("<b>d) Was passiert wenn S = R = 1?</b>"),
    sr_konflikt,
    out_22
]))


## Aufgabe 2.3: Individuelle Berechnung *(4 Punkte)*

Berechnen Sie die **minimale Bandgeschwindigkeit** damit ein Werkstück  
den Abstand zwischen Sensor B1 und Weiche W1 in der vorgegebenen Zeit durchläuft.


In [ ]:
import math as _m

out_23_params = widgets.Output()

v_min_input = widgets.FloatText(
    value=0.0,
    description='v_min (cm/s):',
    style={'description_width':'initial'},
    layout={'width':'260px'}
)
out_23 = widgets.Output()

registriere(
    [v_min_input], out_23,
    check_fn   = lambda: abs(v_min_input.value - v_loesung) < 0.5,
    hinweis_fn = lambda: f"Hinweis: v = s / t = {_s} cm / {_t} s"
)
display(widgets.VBox([
    widgets.Label("Berechnen Sie die minimale Bandgeschwindigkeit:"),
    out_23_params,
    v_min_input, out_23
]))


---
# Handlungsschritt 3: Entscheiden
### Technologieauswahl & Komponentenentscheidung

---


## Aufgabe 3.1: VPS vs. SPS *(6 Punkte)*

Die Anlage soll künftig auch **Werkstücke nach Farbe** unterscheiden  
und die Sortierlogik soll sich bei Bedarf ändern lassen.

Vergleichen Sie verbindungs- und speicherprogrammierte Steuerung  
unter den Gesichtspunkten **Flexibilität, Kosten und Sicherheit**.


In [ ]:
konzept_w = widgets.ToggleButtons(
    options=[('VPS – Verdrahtete Steuerung', 'VPS'),
             ('SPS – Speicherprogrammierbare Steuerung', 'SPS')],
    button_style='info',
    style={'description_width':'initial'}
)
begruendung_31 = widgets.Textarea(
    placeholder='Begründen Sie: Flexibilität? Änderungsaufwand? Kosten bei Erweiterung?',
    description='Begründung:',
    style={'description_width':'initial'},
    layout={'width':'90%','height':'90px'}
)
out_31 = widgets.Output()

registriere(
    [konzept_w, begruendung_31], out_31,
    check_fn   = lambda: h(konzept_w.value or '') == _H_SPS,
    hinweis_fn = lambda: ("Bei einer VPS müsste bei jeder Logikänderung die Verdrahtung "
                          "physisch geändert werden – das ist bei variabler Sortierlogik nicht praktikabel.")
)
display(widgets.VBox([
    widgets.Label("Aufgabe 3.1: Welches Konzept wählen Sie?"),
    konzept_w, begruendung_31, out_31
]))


## Aufgabe 3.2: Sensor-Merkmale zuordnen *(6 Punkte)*

Die drei Sensoren der Anlage sind im Datenblatt wie folgt beschrieben:

- **Lichtschranke B1:** Reflexions-Lichtschranke, Schaltausgang PNP, 24V DC
- **Größensensor B2:** Induktiver Näherungsschalter, Schaltausgang PNP, 24V DC  
- **Farbsensor B3:** RGB-Farbsensor, Analogausgang 0–10V, 24V DC

Ordnen Sie jedem Sensor **genau 2 korrekte** technische Merkmale zu.  
**Achtung:** Einige Aussagen sind falsch oder für diese Anlage nicht zutreffend.


In [ ]:
alle_merkmale = [
    ("--- wählen ---",                                              None),
    ("Optisches Messprinzip – Lichtstrahl wird unterbrochen",       "OPT"),
    ("Induktives Messprinzip – erkennt metallische Objekte",        "IND"),
    ("Wertet RGB-Farbspektrum aus",                                 "RGB"),
    ("Digitaler Schaltausgang (0 oder 1)",                          "DIG"),
    ("Analoger Spannungsausgang (0–10V)",                           "ANAL"),
    ("Benötigt zwingend einen Koppelrelais am SPS-Eingang",         "WRONG1"),
    ("Misst Temperatur des Werkstücks automatisch mit",             "WRONG2"),
    ("Schaltet direkt den 230V-Lastkreis",                          "WRONG3"),
    ("Berührungslose Erkennung – kein Verschleiß",                  "BERUEHR"),
]

loesung_32 = {
    "Lichtschranke B1 (Reflexions-Lichtschranke, PNP)":    {"OPT", "DIG",    "BERUEHR"},
    "Größensensor B2 (Induktiver Näherungsschalter, PNP)": {"IND", "DIG",    "BERUEHR"},
    "Farbsensor B3 (RGB-Sensor, Analogausgang 0-10V)":     {"RGB", "ANAL",   "BERUEHR"},
}
sensoren_32 = list(loesung_32.keys())
sensor_dds_32 = {}
rows_32 = []
for s in sensoren_32:
    dds = []
    for i in range(2):
        dd = widgets.Dropdown(
            options=alle_merkmale,
            layout={'width':'400px', 'margin':'2px'}
        )
        dds.append(dd)
    sensor_dds_32[s] = dds
    rows_32.append(widgets.VBox([
        widgets.HTML(f"<b style='font-size:.9em;'>{s}:</b>"),
        *dds,
        widgets.HTML("<div style='height:6px;'></div>")
    ]))

out_32 = widgets.Output()
all_dds_32 = [dd for dds in sensor_dds_32.values() for dd in dds]

def check_32():
    for s, dds in sensor_dds_32.items():
        vals = {dd.value for dd in dds if dd.value}
        if not (len(vals) == 2 and vals.issubset(loesung_32[s])):
            return False
    return True

def hinweis_32():
    falsch = [s for s, dds in sensor_dds_32.items()
              if not {dd.value for dd in dds if dd.value}.issubset(loesung_32[s])]
    return "Überprüfen Sie: " + "; ".join(falsch) + "."

registriere(all_dds_32, out_32, check_fn=check_32, hinweis_fn=hinweis_32)
display(widgets.VBox([
    widgets.Label("Je 2 korrekte Merkmale zuordnen – Achtung: Einige Aussagen sind falsch oder für diese Anlage nicht zutreffend:"),
    *rows_32,
    out_32
]))


## Aufgabe 3.3: Steuerungslogik erweitern *(10 Punkte)*

Der Farbsensor B3 (E0.2) ist **physisch bereits verbaut** – er liefert einen  
Analogwert 0–10V. Die SPS wertet diesen über einen Schwellenwert aus:  
**E0.2 = 1 → rot**, **E0.2 = 0 → blau**.

Zusätzlich wurde **Größensensor B4 (E0.3)** verbaut, der ausschließlich bei  
**großen** Werkstücken auf 1 schaltet. Beobachten Sie dies am Leitstand.

Die neue Sortierlogik mit den zugehörigen Sensorzuständen:

| Größe | Farbe | Weiche | Ausgang | E0.1 (B2) | E0.3 (B4) | E0.2 (B3) |
|---|---|---|---|---|---|---|
| klein | beliebig | W1 | A0.1 | 0 | 0 | * |
| mittel | rot | W2 | A0.2 | 1 | 0 | 1 |
| mittel | blau | W3 | A0.3 | 1 | 0 | 0 |
| groß | beliebig | W3 | A0.3 | 1 | 1 | * |

**a)** Schreiben Sie die Schaltbedingung für **Weiche W2** (mittel + rot).  
**b)** Schreiben Sie die Schaltbedingung für **Weiche W3** (groß ODER mittel+blau).  
**c)** Klicken Sie auf „Signalflussplan generieren" – prüfen Sie Ihre Gleichungen.

In [ ]:
gleichung_w2 = widgets.Text(
    placeholder='A0.2 = ...',
    description='a) W2 (A0.2):',
    style={'description_width':'120px'},
    layout={'width':'500px'}
)
gleichung_w3 = widgets.Text(
    placeholder='A0.3 = ...',
    description='b) W3 (A0.3):',
    style={'description_width':'120px'},
    layout={'width':'500px'}
)
out_33 = widgets.Output()
out_33_hinweis = widgets.Output()

def check_w2():
    g = gleichung_w2.value.replace(' ','').upper()
    # Korrekt: A0.2 = E0.1 AND NOT E0.3 AND E0.2
    return ('E0.1' in g and 'NOT' in g and 'E0.3' in g and 'E0.2' in g) or \
           ('B2' in g and 'NOT' in g and 'B4' in g and 'B3' in g)

def check_w3():
    g = gleichung_w3.value.replace(' ','').upper()
    # Korrekt: A0.3 = E0.3 OR (E0.1 AND NOT E0.3 AND NOT E0.2)
    return ('NOT' in g and 'E0.3' in g and 'E0.1' in g and 'E0.2' in g) or \
           ('NOT' in g and 'B4' in g and 'B2' in g and 'B3' in g)

registriere(
    [gleichung_w2, gleichung_w3], out_33,
    check_fn   = lambda: check_w2() and check_w3(),
    hinweis_fn = lambda: " ".join(filter(None, [
        "a) W2: Mittelgroße (E0.1=1, E0.3=0) UND rote (E0.2=1) Werkstücke -> A0.2 = E0.1 AND NOT E0.3 AND E0.2." if not check_w2() else "",
        "b) W3: Große Werkstücke (E0.3=1) ODER mittlere blaue (E0.1=1, E0.3=0, E0.2=0) -> A0.3 = E0.3 OR (E0.1 AND NOT E0.3 AND NOT E0.2)." if not check_w3() else "",
    ]))
)
display(widgets.VBox([
    widgets.HTML(
        "<div style='background:#f0f4f8;border-left:4px solid #0077b6;"
        "padding:10px;border-radius:4px;margin-bottom:10px;font-size:.9em;'>"
        "<b>Notation:</b> Schreiben Sie die Gleichungen mit AND, OR, NOT ausgeschrieben.<br>"
        "Beispiel (andere Anlage): <code>Y1 = X1 AND X2</code> &nbsp;|&nbsp; "
        "<code>Y2 = X1 AND NOT X3</code> &nbsp;|&nbsp; "
        "<code>Y3 = X1 AND (X2 OR X3)</code>"
        "</div>"
    ),
    gleichung_w2,
    gleichung_w3,
    out_33,
    out_33_hinweis
]))


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

out_fluss = widgets.Output()

def zeichne_gatter(ax, titel, struktur):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_facecolor('#fafafa')
    ax.set_title(titel, fontsize=9.5, fontweight='bold', pad=8)
    for el in struktur:
        t = el['typ']
        if t == 'label':
            ax.plot(el['x'], el['y'], 'o', color=el['farbe'], markersize=8, zorder=5)
            ax.text(el['x']-0.12, el['y'], el['text'],
                    ha='right', va='center', fontsize=8.5, color=el['farbe'])
        elif t in ('AND', 'OR'):
            x,y,w,h = el['x'],el['y'],el.get('w',1.2),el.get('h',1.6)
            fc = '#dbeafe' if t=='AND' else '#fef9c3'
            ec = '#1d4ed8' if t=='AND' else '#b45309'
            ax.add_patch(mpatches.FancyBboxPatch(
                (x, y-h/2), w, h, boxstyle="round,pad=0.05",
                facecolor=fc, edgecolor=ec, linewidth=2, zorder=3))
            ax.text(x+w/2, y, t, ha='center', va='center',
                    fontsize=9, fontweight='bold', color=ec, zorder=4)
        elif t == 'linie':
            x0,y0 = el['von']; x1,y1 = el['zu']
            ax.annotate('', xy=(x1,y1), xytext=(x0,y0),
                arrowprops=dict(arrowstyle='-',
                color=el.get('farbe','#444'), lw=1.8), zorder=2)
        elif t == 'pfeil':
            ax.annotate('', xy=el['zu'], xytext=el['von'],
                arrowprops=dict(arrowstyle='->', color='#222', lw=2.0), zorder=2)
            if 'text' in el:
                ax.text(el['zu'][0]+0.12, el['zu'][1], el['text'],
                        ha='left', va='center', fontsize=9, fontweight='bold')
        elif t == 'not_kreis':
            ax.plot(el['x'], el['y'], 'o',
                    markerfacecolor='#111', markeredgecolor='#111',
                    markeredgewidth=1.5, markersize=10, zorder=6)

STRUKTUR_W2 = [
    {'typ':'label','x':1.0,'y':6.5,'text':'B1  E0.0','farbe':'#1d4ed8'},
    {'typ':'label','x':1.0,'y':4.5,'text':'B2  E0.1','farbe':'#15803d'},
    {'typ':'label','x':1.0,'y':3.0,'text':'B3  E0.2','farbe':'#7c3aed'},
    {'typ':'linie','von':(1.0,6.5),'zu':(3.5,6.5),'farbe':'#1d4ed8'},
    {'typ':'linie','von':(1.0,4.5),'zu':(3.5,4.5),'farbe':'#15803d'},
    {'typ':'linie','von':(1.0,3.0),'zu':(3.3,3.0),'farbe':'#7c3aed'},
    {'typ':'not_kreis','x':3.35,'y':3.0},
    {'typ':'AND','x':3.5,'y':4.5,'w':1.4,'h':5.2},
    {'typ':'pfeil','von':(4.9,4.5),'zu':(6.8,4.5),'text':'-> A0.2'},
]

STRUKTUR_W3 = [
    {'typ':'label','x':0.8,'y':5.7,'text':'B1  E0.0',  'farbe':'#1d4ed8'},
    {'typ':'label','x':0.8,'y':4.2,'text':'B2  E0.1',  'farbe':'#15803d'},
    {'typ':'label','x':0.8,'y':3.0,'text':'B2  E0.1',  'farbe':'#15803d'},
    {'typ':'label','x':0.8,'y':1.5,'text':'B3  E0.2',  'farbe':'#7c3aed'},
    {'typ':'linie','von':(0.8,4.2),'zu':(4.38,4.2),'farbe':'#15803d'},
    {'typ':'not_kreis','x':4.42,'y':4.2},
    {'typ':'linie','von':(0.8,3.0),'zu':(2.8,3.0),'farbe':'#15803d'},
    {'typ':'linie','von':(0.8,1.5),'zu':(2.8,1.5),'farbe':'#7c3aed'},
    {'typ':'AND','x':2.8,'y':2.25,'w':1.1,'h':2.0},
    {'typ':'linie','von':(3.9,2.25),'zu':(4.5,2.25),'farbe':'#555'},
    {'typ':'OR','x':4.5,'y':3.5,'w':1.2,'h':3.0},
    {'typ':'linie','von':(5.7,3.5),'zu':(6.5,3.5),'farbe':'#555'},
    {'typ':'linie','von':(0.8,5.7),'zu':(6.5,5.7),'farbe':'#1d4ed8'},
    {'typ':'AND','x':6.5,'y':4.5,'w':1.2,'h':3.2},
    {'typ':'pfeil','von':(7.7,4.5),'zu':(9.2,4.5),'text':'-> A0.3'},
]

def zeichne_signalfluss(_):
    g_w2 = gleichung_w2.value.strip()
    g_w3 = gleichung_w3.value.strip()
    with out_fluss:
        out_fluss.clear_output(wait=True)
        if not g_w2 or not g_w3:
            print("Bitte zuerst beide Gleichungen eingeben, dann erneut klicken.")
            return
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        fig.patch.set_facecolor('#fafafa')
        fig.suptitle('Signalflussplan – Weichenlogik nach Erweiterung',
                     fontsize=12, fontweight='bold')
        zeichne_gatter(axes[0], 'W2 | Ihre Gleichung: ' + g_w2, STRUKTUR_W2)
        zeichne_gatter(axes[1], 'W3 | Ihre Gleichung: ' + g_w3, STRUKTUR_W3)
        plt.tight_layout()
        import io as _io,base64 as _b64
        buf=_io.BytesIO()
        fig.savefig(buf,format='png',dpi=100,bbox_inches='tight')
        buf.seek(0); img_b64=_b64.b64encode(buf.read()).decode()
        plt.close(fig)
        display(HTML(f'<img src="data:image/png;base64,{img_b64}" style="max-width:100%;"/>'))

    # Nach erstem Generieren: Gleichungen sperren + Hinweis
    gleichung_w2.disabled = True
    gleichung_w3.disabled = True
    btn_fluss.disabled    = True
    btn_fluss.description = '✅ Schaltplan generiert'
    btn_fluss.button_style = 'success'

    with out_33_hinweis:
        out_33_hinweis.clear_output(wait=True)
        display(HTML(
            "<div style='background:#fff3cd;border-left:4px solid #ffc107;"
            "padding:8px 12px;border-radius:4px;margin-top:6px;font-size:.88em;'>"
            "🔒 <b>Gleichungen gesperrt</b> – der Signalflussplan wurde auf Basis "
            "Ihrer Eingaben generiert und ist nun fest. "
            "Vergleichen Sie den Plan mit Ihren Gleichungen.</div>"
        ))

btn_fluss = widgets.Button(
    description="Signalflussplan generieren",
    button_style='primary', icon='bolt',
    layout={'width':'260px'}
)
btn_fluss.on_click(zeichne_signalfluss)
_alle_widgets.append(btn_fluss)

display(widgets.VBox([
    widgets.HTML(
        "<div style='background:#e8f4fd;border-left:4px solid #0077b6;"
        "padding:10px;border-radius:4px;margin-bottom:8px;'>"
        "<b>c) Signalflussplan</b> – erst Gleichungen unter a) und b) eingeben, "
        "dann hier generieren. Die Gleichungen werden danach gesperrt.</div>"
    ),
    btn_fluss, out_fluss
]))


---
# Handlungsschritt 4: Durchführen
### Virtuelle Inbetriebnahme – Fehlerdiagnose am Leitstand

Die Anlage wurde geliefert – zeigt aber **Fehlverhalten**.  
Der Fehler-Leitstand unten zeigt genau das was Sie vorfinden.

> Beobachten Sie den Fehler-Leitstand, analysieren Sie das Fehlverhalten  
> und beantworten Sie die Aufgaben. **Der obere Leitstand bleibt als Referenz.**

---


## Aufgabe 4.1: Fehlerdiagnose – Weichenlogik *(12 Punkte)*

Beobachten Sie den **Fehler-Leitstand** unten: Geben Sie verschiedene  
Werkstücktypen auf und notieren Sie welche Weiche immer öffnet.

**a)** Beschreiben Sie das Fehlverhalten präzise.  
**b)** Identifizieren Sie welches Eingangsbit nicht ausgewertet wird.  
**c)** Wählen Sie die korrekte Korrekturmaßnahme.


In [ ]:
fehler_html = r'''
<div id="fls" style="font-family:'Segoe UI',sans-serif;background:#0d1117;color:#c9d1d9;
     border:2px solid #21262d;border-radius:12px;max-width:940px;
     box-shadow:0 0 40px rgba(0,100,255,.1);overflow:hidden;">
  <div style="background:linear-gradient(90deg,#0d1f3c,#0a2a50);padding:10px 20px;
              display:flex;align-items:center;gap:14px;border-bottom:1px solid #21262d;">
    <div style="width:12px;height:12px;border-radius:50%;background:#da3633;
         box-shadow:0 0 8px #da3633;"></div>
    <div>
      <div style="font-size:1em;font-weight:700;color:#58a6ff;">
        LEITSTAND – SORTIERANLAGE SA-1 (DIAGNOSE)</div>
      <div style="font-size:.72em;color:#8b949e;font-family:monospace;">
        BETRIEBSZUSTAND: AUS | BAND: GESTOPPT</div>
    </div>
    <div style="margin-left:auto;font-family:monospace;font-size:.85em;color:#3fb950;"
         id="fws_cnt">WS gesamt: 0</div>
  </div>
  <div style="padding:16px 20px 8px;">
  <svg viewBox="0 0 880 270" xmlns="http://www.w3.org/2000/svg"
       style="width:100%;border-radius:8px;background:#161b22;">
    <!-- Band -->
    <rect x="55" y="128" width="710" height="28" rx="4" fill="#21262d" stroke="#30363d" stroke-width="1.5"/>
    <rect id="fband" x="55" y="131" width="710" height="10" rx="2" fill="#1f6feb" opacity="0.0"/>
    <circle cx="68"  cy="142" r="14" fill="#30363d" stroke="#484f58" stroke-width="2"/>
    <circle cx="750" cy="142" r="14" fill="#30363d" stroke="#484f58" stroke-width="2"/>
    <!-- WS -->
    <rect id="fws_r" x="75" y="112" width="36" height="24" rx="5"
          fill="#f78166" stroke="#ff7b72" stroke-width="1.5" opacity="0"/>
    <text id="fws_l" x="93" y="128" text-anchor="middle" font-size="8"
          fill="white" font-weight="bold" opacity="0"></text>
    <!-- B1 -->
    <line x1="200" y1="106" x2="200" y2="158" stroke="#388bfd" stroke-width="1.5"
          stroke-dasharray="4,3" opacity="0.4"/>
    <rect id="fb1b" x="185" y="96" width="30" height="16" rx="3"
          fill="#21262d" stroke="#388bfd" stroke-width="1.5"/>
    <text x="200" y="108" text-anchor="middle" font-size="8" fill="#388bfd" font-weight="bold">B1</text>
    <text x="200" y="173" text-anchor="middle" font-size="7" fill="#6e7681">Lichtschranke</text>
    <text x="200" y="183" text-anchor="middle" font-size="7" fill="#6e7681">E 0.0</text>
    <!-- B2 -->
    <line x1="390" y1="106" x2="390" y2="158" stroke="#3fb950" stroke-width="1.5"
          stroke-dasharray="4,3" opacity="0.4"/>
    <rect id="fb2b" x="375" y="96" width="30" height="16" rx="3"
          fill="#21262d" stroke="#3fb950" stroke-width="1.5"/>
    <text x="390" y="108" text-anchor="middle" font-size="8" fill="#3fb950" font-weight="bold">B2</text>
    <text x="390" y="173" text-anchor="middle" font-size="7" fill="#6e7681">Größensensor</text>
    <text x="390" y="183" text-anchor="middle" font-size="7" fill="#6e7681">E 0.1</text>
    <!-- B4 -->
    <line x1="475" y1="106" x2="475" y2="158" stroke="#ffa657" stroke-width="1.5"
          stroke-dasharray="4,3" opacity="0.4"/>
    <rect id="fb4b" x="460" y="96" width="30" height="16" rx="3"
          fill="#21262d" stroke="#ffa657" stroke-width="1.5"/>
    <text x="475" y="108" text-anchor="middle" font-size="8" fill="#ffa657" font-weight="bold">B4</text>
    <text x="475" y="173" text-anchor="middle" font-size="7" fill="#6e7681">Gr&#246;&#223;ens. 2</text>
    <text x="475" y="183" text-anchor="middle" font-size="7" fill="#6e7681">E 0.3</text>
    <!-- B3 -->
    <line x1="560" y1="106" x2="560" y2="158" stroke="#d2a8ff" stroke-width="1.5"
          stroke-dasharray="4,3" opacity="0.4"/>
    <rect id="fb3b" x="545" y="96" width="30" height="16" rx="3"
          fill="#21262d" stroke="#d2a8ff" stroke-width="1.5"/>
    <text x="560" y="108" text-anchor="middle" font-size="8" fill="#d2a8ff" font-weight="bold">B3</text>
    <text x="560" y="173" text-anchor="middle" font-size="7" fill="#6e7681">Farbsensor</text>
    <text x="560" y="183" text-anchor="middle" font-size="7" fill="#6e7681">E 0.2</text>
    <!-- Weichen – W1 immer aktiv (Fehlverhalten), W2/W3 inaktiv -->
    <polygon id="fw1p" points="630,156 670,156 660,205 620,205"
             fill="#e3b341" opacity="0.9" stroke="#d29922" stroke-width="2"/>
    <text x="645" y="186" text-anchor="middle" font-size="9" fill="#0d1117" font-weight="bold">W1</text>
    <text x="645" y="220" text-anchor="middle" font-size="7" fill="#e3b341">klein · A0.1</text>
    <polygon points="675,156 715,156 705,205 665,205"
             fill="#21262d" opacity="0.4" stroke="#30363d" stroke-width="1.5"/>
    <text x="690" y="186" text-anchor="middle" font-size="9" fill="#484f58" font-weight="bold">W2</text>
    <text x="690" y="220" text-anchor="middle" font-size="7" fill="#484f58">mittel · A0.2</text>
    <polygon points="720,156 760,156 750,205 710,205"
             fill="#21262d" opacity="0.4" stroke="#30363d" stroke-width="1.5"/>
    <text x="735" y="186" text-anchor="middle" font-size="9" fill="#484f58" font-weight="bold">W3</text>
    <text x="735" y="220" text-anchor="middle" font-size="7" fill="#484f58">groß · A0.3</text>
    <!-- NOT-HALT -->
    <rect id="fnot" x="14" y="196" width="52" height="32" rx="4"
          fill="#da3633" stroke="#f85149" stroke-width="2" style="cursor:pointer"/>
    <text x="40" y="209" text-anchor="middle" font-size="8" fill="white" font-weight="bold">NOT-</text>
    <text x="40" y="221" text-anchor="middle" font-size="8" fill="white" font-weight="bold">HALT</text>
    <!-- SPS -->
    <rect x="14" y="14" width="140" height="72" rx="6" fill="#161b22" stroke="#30363d" stroke-width="1.5"/>
    <text x="84" y="30" text-anchor="middle" font-size="8.5" fill="#58a6ff" font-weight="bold">SPS – Steuereinheit</text>
    <text id="fe_disp" x="84" y="46" text-anchor="middle" font-size="7.5"
          fill="#6e7681" font-family="monospace">E: 0 0 0  0 0 0</text>
    <text id="fa_disp" x="84" y="59" text-anchor="middle" font-size="7.5"
          fill="#6e7681" font-family="monospace">A: 0 0 0  0 0</text>
    <text id="fprog" x="84" y="74" text-anchor="middle" font-size="7" fill="#3fb950">Programm: BEREIT</text>
    <!-- Info-Box -->
    <rect x="762" y="14" width="104" height="52" rx="4" fill="#161b22" stroke="#30363d" stroke-width="1"/>
    <text x="814" y="30" text-anchor="middle" font-size="7" fill="#6e7681">Letztes Werkstück</text>
    <text id="flast" x="814" y="54" text-anchor="middle" font-size="13"
          fill="#c9d1d9" font-family="monospace" font-weight="bold">-</text>
    <rect x="762" y="72" width="104" height="42" rx="4" fill="#161b22" stroke="#30363d" stroke-width="1"/>
    <text x="814" y="88" text-anchor="middle" font-size="7" fill="#6e7681">Fehlercode</text>
    <text id="ferr" x="814" y="107" text-anchor="middle" font-size="11"
          fill="#3fb950" font-family="monospace">OK</text>
  </svg>
  </div>
  <div style="padding:4px 20px 14px;display:flex;gap:8px;flex-wrap:wrap;align-items:center;">
    <button onclick="fStart()" style="background:#238636;color:#fff;border:none;
      border-radius:6px;padding:7px 18px;cursor:pointer;font-weight:600;font-size:.88em;">
      ANLAUF</button>
    <button onclick="fStop()" style="background:#da3633;color:#fff;border:none;
      border-radius:6px;padding:7px 18px;cursor:pointer;font-weight:600;font-size:.88em;">
      STOP</button>
    <select id="ftyp" style="background:#21262d;color:#c9d1d9;border:1px solid #30363d;
      border-radius:6px;padding:5px 10px;font-size:.83em;">
      <option value="klein_rot">Klein - Rot</option>
      <option value="gross_blau">Groß - Blau</option>
      <option value="mittel_rot">Mittel - Rot</option>
      <option value="gross_rot">Groß - Rot</option>
    </select>
    <button onclick="fSende()" style="background:#1f6feb;color:#fff;border:none;
      border-radius:6px;padding:7px 14px;cursor:pointer;font-size:.83em;">
      + Werkstück aufgeben</button>
  </div>
</div>
<script>
var fAn=false,fWsAktiv=false,fX=75,fGes=0,fId=null,fNotHalt=false,fTyp='klein_rot',fB4=0;
var fFarben={'klein_rot':'#f78166','gross_blau':'#79c0ff',
             'mittel_rot':'#ffa657','gross_rot':'#ff7b72'};
var fBr={'klein':30,'mittel':44,'gross':60};

function fUpSPS(e,a){
  document.getElementById('fe_disp').textContent='E: '+e.slice(0,3).join(' ')+'  '+e.slice(3).join(' ');
  document.getElementById('fa_disp').textContent='A: '+a.slice(0,3).join(' ')+'  '+a.slice(3).join(' ');
}
function fStart(){
  if(fNotHalt) return;
  fAn=true;
  document.getElementById('fband').style.opacity='0.5';
  document.getElementById('fprog').textContent='Programm: AKTIV';
  fUpSPS([0,0,0,0,0,0],[1,0,0,0,0]);
  if(!fId) fId=setInterval(fStep,35);
}
function fStop(){
  fAn=false;
  document.getElementById('fband').style.opacity='0.0';
  document.getElementById('fprog').textContent='Programm: BEREIT';
  fUpSPS([0,0,0,0,0,0],[0,0,0,0,0]);
}
function fSende(){
  if(!fAn||fWsAktiv) return;
  fTyp=document.getElementById('ftyp').value;
  var t=fTyp.split('_'),gr=t[0];
  var w=document.getElementById('fws_r'),l=document.getElementById('fws_l');
  var br=fBr[gr];
  w.setAttribute('width',br);
  w.setAttribute('fill',fFarben[fTyp]);
  l.textContent=gr.charAt(0).toUpperCase()+'/'+t[1].charAt(0).toUpperCase();
  w.setAttribute('x',75); l.setAttribute('x',75+br/2);
  w.style.opacity='0.92'; l.style.opacity='1';
  fX=75; fWsAktiv=true; fGes++; fB4=0;
  document.getElementById('fws_cnt').textContent='WS gesamt: '+fGes;
  document.getElementById('ferr').textContent='OK';
  document.getElementById('ferr').style.color='#3fb950';
}
function fBlink(id,col){
  var el=document.getElementById(id);
  el.setAttribute('fill',col);
  el.style.filter='drop-shadow(0 0 7px '+col+')';
}
function fDim(id){
  var el=document.getElementById(id);
  el.setAttribute('fill','#21262d');
  el.style.filter='none';
}
function fStep(){
  if(!fAn||!fWsAktiv) return;
  fX+=3;
  var t=fTyp.split('_'),gr=t[0];
  var br=fBr[gr];
  var w=document.getElementById('fws_r'),l=document.getElementById('fws_l');
  w.setAttribute('x',fX); l.setAttribute('x',fX+br/2);
  if(fX>188&&fX<225){ fBlink('fb1b','#388bfd'); fUpSPS([1,0,0,0,0,0],[1,0,0,0,0]); }
  else fDim('fb1b');
  // B2 leuchtet aber wird NICHT ausgewertet (Fehlverhalten)
  if(fX>378&&fX<418){ fBlink('fb2b','#3fb950'); fUpSPS([1,1,0,0,0,0],[1,0,0,0,0]); }
  else fDim('fb2b');
  // B4 leuchtet für groß, wird aber NICHT ausgewertet (Fehlverhalten)
  if(fX>463&&fX<500){
    fBlink('fb4b','#ffa657');
    fB4=(gr==='gross')?1:0;
    fUpSPS([1,1,0,fB4,0,0],[1,0,0,0,0]);
  } else { if(fX<463){fB4=0;} fDim('fb4b'); }
  if(fX>548&&fX<587){ fBlink('fb3b','#d2a8ff'); fUpSPS([1,1,1,fB4,0,0],[1,0,0,0,0]); }
  else fDim('fb3b');
  // Fehler: W1 öffnet IMMER unabhängig von Größe
  if(fX>622&&fX<668){
    document.getElementById('fw1p').style.filter='drop-shadow(0 0 10px #e3b341)';
    document.getElementById('flast').textContent='W1 ('+gr+')';
    document.getElementById('ferr').textContent='--';
    document.getElementById('ferr').style.color='#8b949e';
    fWsAktiv=false;
    w.style.opacity='0'; l.style.opacity='0';
    fUpSPS([0,0,0,0,0,0],[1,1,0,0,0]);
    setTimeout(function(){document.getElementById('fw1p').style.filter='none';},900);
    return;
  }
  if(fX>780){ fWsAktiv=false; w.style.opacity='0'; l.style.opacity='0'; }
}
</script>
'''
display(HTML(fehler_html))


In [ ]:
beob_41 = widgets.Textarea(
    placeholder='a) Was passiert mit jedem Werkstück? Welche Weiche öffnet immer?'
                ' Welche Weichen öffnen nie?',
    description='a) Beobachtung:',
    style={'description_width':'140px'},
    layout={'width':'90%', 'height':'80px'}
)
fehler_bit = widgets.RadioButtons(
    value=None,
    options=[
        ("E0.0 – Lichtschranke B1 wird nicht ausgewertet", "E00"),
        ("E0.1 – Größensensor B2 wird nicht ausgewertet",  "E01"),
        ("E0.2 – Farbsensor B3 steuert die Weiche",        "E02"),
        ("A0.0 – Förderband-Ausgang ist falsch belegt",     "A00"),
    ],
    description='b) Fehlerursache:',
    style={'description_width':'initial'},
    layout={'width':'700px'}
)
korrektur_41 = widgets.RadioButtons(
    value=None,
    options=[
        ("E0.1 (Größensensor B2) mit der Weichenauswahl verknüpfen", "KORREKT"),
        ("Sensor B1 (E0.0) durch einen anderen Typ ersetzen",         "FALSCH1"),
        ("Alle drei Weichen auf Ausgang A0.1 legen",                  "FALSCH2"),
        ("Farbsensor B3 (E0.2) deaktivieren",                        "FALSCH3"),
        ("SPS komplett neu programmieren und alle Adressen tauschen", "FALSCH4"),
    ],
    description='c) Korrektur:',
    style={'description_width':'initial'},
    layout={'width':'700px'}
)
out_41 = widgets.Output()

_alle_widgets.extend([beob_41, fehler_bit, korrektur_41])
registriere(
    [fehler_bit, korrektur_41], out_41,
    check_fn   = lambda: (h(fehler_bit.value or '') == _H_E01 and h(korrektur_41.value or '') == _H_KORREKT),
    hinweis_fn = lambda: " ".join(filter(None, [
        "b) B2 meldet die Größe über E0.1 – dieses Bit steuert die Weichenauswahl." if fehler_bit.value != 'E01' else "",
        "c) Nur E0.1 muss korrekt mit der Weichenlogik verknüpft werden." if korrektur_41.value != 'KORREKT' else "",
    ]))
)
display(widgets.VBox([beob_41, fehler_bit, korrektur_41, out_41]))


## Aufgabe 4.2: Normen & Vorschriften – NOT-HALT *(8 Punkte)*

Der NOT-HALT der Anlage entspricht **nicht der DIN EN ISO 13850**.  
Der Taster ist als **Schließer** (Öffner bei Betätigung) verdrahtet.

**a)** Erklären Sie warum ein NOT-HALT zwingend als **Öffner** ausgeführt sein muss.  
**b)** Nennen Sie die Kategorie des Sicherheitsstopps (Stopp-Kategorie 0 oder 1).  
**c)** Welche Farbe schreibt die Norm für NOT-HALT-Taster vor?


In [ ]:
oeffner_w = widgets.RadioButtons(
    value=None,
    options=[
        ("Öffner (Ruhestromprinzip: sicher AUS bei Leitungsbruch)", "OEFFNER"),
        ("Schließer (einfacher zu verdrahten)",                      "SCHLIESSER"),
        ("Wechsler (flexibel einsetzbar)",                           "WECHSLER"),
    ],
    description='a) Kontaktart:',
    style={'description_width':'120px'},
    layout={'width':'580px'}
)
stopp_kat = widgets.RadioButtons(
    value=None,
    options=[
        ("Kat. 0 – sofortiger Energieentzug am Aktor",    "KAT0"),
        ("Kat. 1 – gesteuerter Halt, dann Energieentzug", "KAT1"),
        ("Kat. 2 – Halt, Motor bleibt bestromt",          "KAT2"),
    ],
    description='b) Stopp-Kat.:',
    style={'description_width':'120px'},
    layout={'width':'580px'}
)
farbe_w = widgets.RadioButtons(
    value=None,
    options=[
        ("Roter Pilztaster auf GELBEM Hintergrund", "KORREKT"),
        ("Roter Pilztaster auf rotem Hintergrund",  "FALSCH1"),
        ("Gelber Taster auf schwarzem Hintergrund", "FALSCH2"),
    ],
    description='c) Farbe/Norm:',
    style={'description_width':'120px'},
    layout={'width':'580px'}
)
begruendung_42 = widgets.Textarea(
    placeholder='a) Begründen Sie das Ruhestromprinzip: Was passiert bei Kabelbruch '
                'wenn der NOT-HALT-Taster als Schließer verdrahtet wäre?',
    description='Begründung:',
    style={'description_width':'120px'},
    layout={'width':'90%', 'height':'80px'}
)
out_42 = widgets.Output()

registriere(
    [oeffner_w, stopp_kat, farbe_w, begruendung_42], out_42,
    check_fn   = lambda: (h(stopp_kat.value or '') == _H_KAT0 and h(farbe_w.value or '') == _H_KORREKT),
    hinweis_fn = lambda: " ".join(filter(None, [
        "a) Öffner: Ruhestromprinzip – bei Kabelbruch öffnet der Kreis -> sicherer Stopp." if oeffner_w.value != 'OEFFNER' else "",
        "b) NOT-HALT = Stopp-Kategorie 0: sofortiger Energieentzug." if stopp_kat.value != 'KAT0' else "",
        "c) DIN EN ISO 13850: Roter Pilz auf GELBEM Hintergrund." if farbe_w.value != 'KORREKT' else "",
    ]))
)
display(widgets.VBox([begruendung_42, stopp_kat, farbe_w, out_42]))


---
# Handlungsschritt 5: Kontrollieren
### Technische Abnahme

---


## Aufgabe 5.1: Elektrische Prüfung nach DGUV Vorschrift 3 *(10 Punkte)*

Vor der Inbetriebnahme ist eine **Prüfung elektrischer Betriebsmittel**  
nach DGUV Vorschrift 3 (früher BGV A3) vorgeschrieben.

**a)** Nennen Sie **drei Prüfschritte** der Wiederholungsprüfung nach DGUV V3  
in der richtigen Reihenfolge.  
**b)** Was wird bei der **Sichtprüfung** konkret kontrolliert? (2 Punkte)  
**c)** Wie lautet die maximale Prüffrist für ortsveränderliche Betriebsmittel  
in Produktionsbereichen nach DGUV V3?


In [ ]:
pruef_schritte = [
    "Sichtprüfung (Schäden, Verschleiß, Beschriftung)",
    "Messung (Schutzleiterwiderstand, Isolationswiderstand, Ableitstrom)",
    "Funktionsprüfung (Schutzeinrichtungen, NOT-HALT, Verriegelungen)",
]
pruef_opt = [("--- wählen ---", None)] + [(s, s) for s in pruef_schritte]

schritt_1 = widgets.Dropdown(options=pruef_opt, description='Schritt 1:',
                             style={'description_width':'initial'}, layout={'width':'580px'})
schritt_2 = widgets.Dropdown(options=pruef_opt, description='Schritt 2:',
                             style={'description_width':'initial'}, layout={'width':'580px'})
schritt_3 = widgets.Dropdown(options=pruef_opt, description='Schritt 3:',
                             style={'description_width':'initial'}, layout={'width':'580px'})


sichtpruef = widgets.Textarea(
    placeholder='b) Was wird bei der Sichtprüfung konkret kontrolliert? '
                '(z.B. Kabelzustand, Gehäuse, Kennzeichnung...)',
    description='b) Sichtprüfung:',
    style={'description_width':'initial'},
    layout={'width':'90%','height':'70px'}
)

pruef_frist = widgets.RadioButtons(
    value=None,
    options=[("6 Monate", "6M"),
             ("1 Jahr",   "1J"),
             ("2 Jahre",  "2J"),
             ("4 Jahre",  "4J")],
    description='c) Prüffrist:',
    style={'description_width':'initial'}
)
out_51 = widgets.Output()

registriere(
    [schritt_1, schritt_2, schritt_3, pruef_frist], out_51,
    check_fn   = lambda: (
        schritt_1.value == pruef_schritte[0] and
        schritt_2.value == pruef_schritte[1] and
        schritt_3.value == pruef_schritte[2] and
        h(pruef_frist.value or '') == _H_1J
    ),
    hinweis_fn = lambda: " ".join(filter(None,[
        "a) Reihenfolge: Sicht → Messung → Funktion." if not (
            schritt_1.value==pruef_schritte[0] and
            schritt_2.value==pruef_schritte[1] and
            schritt_3.value==pruef_schritte[2]) else "",
        "c) DGUV V3: ortsveränderliche Betriebsmittel in Produktion → 1 Jahr." if pruef_frist.value!='1J' else "",
    ]))
)
_alle_widgets.extend([schritt_1, schritt_2, schritt_3, sichtpruef, pruef_frist])
display(widgets.VBox([
    widgets.Label("a) Ordnen Sie die drei Prüfschritte in der richtigen Reihenfolge:"),
    schritt_1, schritt_2, schritt_3, sichtpruef, pruef_frist, out_51
]))


---
# Handlungsschritt 6: Bewerten
### Prozess- und Selbstreflexion

---


## Aufgabe 6.1: Technische Dokumentation *(6 Punkte)*

Schreiben Sie eine kurze **Funktionsbeschreibung** der Sortieranlage  
in technischer Sprache (3–5 Sätze). Beschreiben Sie den Ablauf vom  
Aufgeben des Werkstücks bis zur Weichenöffnung.


In [ ]:
funk_beschr = widgets.Textarea(
    placeholder='Beispiel: "Nach dem Einschalten des Förderbands durch A0.0 '
                'erfasst Lichtschranke B1 (E0.0) das aufgegebene Werkstück..."',
    description='Beschreibung:',
    style={'description_width':'initial'},
    layout={'width':'95%','height':'130px'}
)
_alle_widgets.append(funk_beschr)
display(widgets.VBox([
    widgets.Label("Aufgabe 6.1: Technische Funktionsbeschreibung:"),
    funk_beschr
]))


## Aufgabe 6.2: Fehleranalyse & Prozessreflexion *(8 Punkte)*

Vergleichen Sie den **Fehler-Leitstand** (Fehlerzustand) mit dem  
**Referenz-Leitstand** (Sollzustand) ganz oben.

**a)** Beschreiben Sie in eigenen Worten: Was war die technische Ursache des Fehlers?  
**b)** Wie hat Ihnen der Fehler-Leitstand geholfen, die Ursache zu identifizieren?  
**c)** Was würden Sie bei einem ähnlichen Inbetriebnahme-Auftrag zuerst prüfen?


In [ ]:
ursache_txt = widgets.Textarea(
    placeholder='a) Beschreiben Sie die technische Ursache: Welches Signal war nicht '
                'korrekt verknüpft? Was war die Folge im Prozess?',
    description='a) Ursache:',
    layout={'width':'95%','height':'90px'}
)
leitstand_nutzen = widgets.Textarea(
    placeholder='b) Wie hat der Fehler-Leitstand bei der Diagnose geholfen? '
                'Was konnten Sie direkt beobachten?',
    description='b) Leitstand:',
    layout={'width':'95%','height':'70px'}
)
massnahme_txt = widgets.Textarea(
    placeholder='c) Was würden Sie beim nächsten Auftrag zuerst prüfen? '
                '(z.B. Sensor-Adressen, Verknüpfungen, Testlauf mit allen Werkstücktypen)',
    description='c) Massnahme:',
    layout={'width':'95%','height':'70px'}
)
erkannt_wie = widgets.RadioButtons(
    value=None,
    options=[("Durch systematischen Test aller Werkstücktypen am Leitstand", "SYST"),
             ("Durch Ablesen der SPS-Fehleranzeige (E-02)", "ANZEIGE"),
             ("Durch zufälliges Ausprobieren", "ZUFALL")],
    description='Wie erkannt:',
    style={'description_width':'120px'},
    layout={'width':'580px'},
)
_alle_widgets.extend([ursache_txt, leitstand_nutzen, massnahme_txt, erkannt_wie])
display(widgets.VBox([
    ursache_txt, leitstand_nutzen, erkannt_wie, massnahme_txt
]))


---
## 📊 Punkteübersicht

| Aufgabe | Inhalt | Art | Punkte |
|---|---|---|---|
| 1.1a | EVA-Analyse (7 Komponenten) | geschlossen | 3,5 |
| 1.1b | SPS-Adressen (7 Felder × 0,5) | manuell | 3,5 |
| 1.2 | Wirkungskette (6 Schritte) | geschlossen | 8 |
| 2.1a | Logische Verknüpfung | geschlossen | 3 |
| 2.1b | Boolesche Gleichung | manuell | 3 |
| 2.1c | Wahrheitstafel | geschlossen | 6 |
| 2.2a | SR-Flipflop: S (Set) | geschlossen | 2 |
| 2.2b | SR-Flipflop: R (Reset) | geschlossen | 2 |
| 2.2c | Erklärung speichernd/nicht-speichernd | manuell | 3 |
| 2.2d | Verhalten S=R=1 | geschlossen | 2 |
| 2.3 | Individuelle Berechnung Bandgeschwindigkeit | geschlossen | 4 |
| 3.1a | VPS vs. SPS – Auswahl | geschlossen | 2 |
| 3.1b | VPS vs. SPS – Begründung | manuell | 4 |
| 3.2 | Sensor-Merkmale zuordnen | geschlossen | 6 |
| 3.3a | Boolesche Gleichung W2 | halb-offen | 4 |
| 3.3b | Boolesche Gleichung W3 | halb-offen | **Bonus +6** |
| 4.1a | Fehlerbeschreibung (Beobachtung) | manuell | 4 |
| 4.1b | Fehlerursache identifizieren | geschlossen | 4 |
| 4.1c | Korrekturmaßnahme | geschlossen | 4 |
| 4.2a | NOT-HALT Begründung | manuell | 4 |
| 4.2b | Stopp-Kategorie | geschlossen | 2 |
| 4.2c | Farbe/Norm DIN EN ISO 13850 | geschlossen | 2 |
| 5.1a | DGUV V3 Prüfschritte (Reihenfolge) | geschlossen | 4 |
| 5.1b | Sichtprüfung beschreiben | manuell | 3 |
| 5.1c | Prüffrist | geschlossen | 3 |
| 6.1 | Technische Funktionsbeschreibung | manuell | 6 |
| 6.2a | Fehlerursache erklären | manuell | 2 |
| 6.2b | Leitstand-Nutzen reflektieren | manuell | 2 |
| 6.2c | Maßnahme für nächsten Auftrag | manuell | 4 |
| **Gesamt** | | | **100** |
| **Bonus** | Boolesche Gleichung W3 korrekt | halb-offen | **+6** |

---

## 🎓 Notenschlüssel

| Note | Punkte | Prozent |
|---|---|---|
| Sehr gut (1) | 100 – 92 | 100 % – 92 % |
| Gut (2) | 91 – 81 | 91 % – 81 % |
| Befriedigend (3) | 80 – 67 | 80 % – 67 % |
| Ausreichend (4) | 66 – 50 | 66 % – 50 % |
| Nicht bestanden (5) | 49 – 20 | 49 % – 20 % |
| Ungenügend (6) | unter 20 | unter 20 % |

*Bonuspunkte (W3) können die Note verbessern, werden aber nicht über 100 angerechnet.*

---



In [ ]:
# ── Manuelle Bewertungsfelder ───────────────

manuelle_felder = [
    ('1.1b', 'SPS-Adressen (7 Felder, je 0,5 Pkt.)',           7),  # x0.5 intern
    ('2.1b', 'Boolesche Gleichung A0.0',                       3),
    ('2.2c', 'Erklaerung speichernd/nicht-speichernd',         3),
    ('3.1',  'VPS vs. SPS Begruendung',                        4),
    ('4.1a', 'Fehlerbeschreibung',                             4),
    ('4.2a', 'NOT-HALT Begruendung',                           4),
    ('5.1b', 'Sichtpruefung beschreiben',                      3),
    ('6.1',  'Funktionsbeschreibung',                          6),
    ('6.2a', 'Fehlerursache',                                  2),
    ('6.2b', 'Leitstand-Nutzen',                               2),
    ('6.2c', 'Massnahme',                                      4),
]

bewertungs_inputs = {}
out_bewertung = widgets.Output()

def erstelle_bewertungsfelder():
    _PWD_HASH = '835d6dc88b708bc6'  # h('admin')
    with out_bewertung:
        out_bewertung.clear_output()
        display(HTML(
            "<div style='background:#f0f4f8;border-left:4px solid #0077b6;"
            "padding:12px;border-radius:6px;margin-top:12px;'>"
            "<b>Manuelle Bewertung (nur für Lehrkraft)</b><br>"
            "<small>Passwort eingeben, um Felder freizuschalten:</small></div>"
        ))
        pwd_w      = widgets.Password(placeholder='Passwort', description='PW:',
                         layout={'width':'220px'}, style={'description_width':'30px'})
        btn_unlock = widgets.Button(description='Entsperren', button_style='warning',
                         icon='unlock', layout={'width':'140px'})
        out_unlock = widgets.Output()
        rows = []
        for aufg, name, max_p in manuelle_felder:
            einheit = ' ×0,5' if aufg == '1.1b' else ' Pkt.'
            sl = widgets.BoundedIntText(
                value=0, min=0, max=max_p,
                description=f'{aufg} – {name} (max. {max_p}{einheit}):',
                style={'description_width': '390px'},
                layout={'width': '540px'},
                disabled=True
            )
            bewertungs_inputs[aufg] = (sl, max_p)
            rows.append(sl)
        btn_berechne = widgets.Button(
            description='Gesamtpunkte berechnen',
            button_style='primary',
            layout={'width': '240px'},
            disabled=True
        )
        def entsperre(b):
            if h(pwd_w.value) == _PWD_HASH:
                for sl, _ in bewertungs_inputs.values(): sl.disabled = False
                btn_berechne.disabled = False; btn_unlock.disabled = True
                btn_unlock.description = 'Entsperrt'; btn_unlock.button_style = 'success'
                pwd_w.disabled = True
                with out_unlock:
                    out_unlock.clear_output()
                    display(HTML("<span style='color:#28a745;font-size:.85em;'>Freigegeben.</span>"))
            else:
                with out_unlock:
                    out_unlock.clear_output()
                    display(HTML("<span style='color:#dc3545;font-size:.85em;'>Falsches Passwort.</span>"))
        btn_unlock.on_click(entsperre)
        out_gesamt = widgets.Output()

        def berechne_gesamt(b):
            auto = 0.0
            # 1.1 EVA-Dropdowns (3,5 Pkt.)
            try:
                if all(eva_ws[k].value == loesung_eva[k] for k in komponenten):
                    auto += 3.5
            except Exception: pass
            # 1.2 Wirkungskette (8 / 4 / 0 Pkt.)
            try:
                if len(wk_auswahl) == len(wk_korrekt):
                    nf = sum(1 for a,c in zip(wk_auswahl, wk_korrekt) if a != c)
                    if nf == 0:   auto += 8
                    elif nf == 1: auto += 4
            except Exception: pass
            # 2.1 a) Logik (3 Pkt.)
            try:
                if logik_w.value == 'AND': auto += 3
            except Exception: pass
            # 2.1 c) Wahrheitstafel (6 Pkt.)
            try:
                if all(wt_eingaben[(x1,x2)].value == wt_loesung_21[(x1,x2)]
                       for x1 in [0,1] for x2 in [0,1]):
                    auto += 6
            except Exception: pass
            # 2.2 a+b+d (je 2 Pkt. = 6 Pkt.)
            try:
                if h(sr_s.value or '') == _H_NOT:     auto += 2
                if h(sr_r.value or '') == _H_QUIT:    auto += 2
                if h(sr_konflikt.value or '') == _H_SET_DOM: auto += 2
            except Exception: pass
            # 2.3 v_min (4 Pkt.)
            try:
                if abs(v_min_input.value - v_loesung) < 0.5: auto += 4
            except Exception: pass
            # 3.1 Auswahl (2 Pkt.)
            try:
                if h(konzept_w.value or '') == _H_SPS: auto += 2
            except Exception: pass
            # 3.2 Sensormerkmale (je 2 Pkt. = 6 Pkt., Teilmenge gültig)
            try:
                for s, dds in sensor_dds_32.items():
                    vals = {dd.value for dd in dds if dd.value}
                    if len(vals) == 2 and vals.issubset(loesung_32[s]):
                        auto += 2
            except Exception: pass
            # 3.3 a) W2 (4 Pkt. Pflicht)
            try:
                if check_w2(): auto += 4
            except Exception: pass
            # 3.3 b) W3 (6 Pkt. Bonus)
            bonus = 0.0
            try:
                if check_w3(): bonus += 6
            except Exception: pass
            # 4.1 b+c (je 4 Pkt. = 8 Pkt.)
            try:
                if h(fehler_bit.value or '') == _H_E01:     auto += 4
                if h(korrektur_41.value or '') == _H_KORREKT: auto += 4
            except Exception: pass
            # 4.2 b+c (je 2 Pkt. = 4 Pkt.)
            try:
                if h(stopp_kat.value or '') == _H_KAT0:    auto += 2
                if h(farbe_w.value or '') == _H_KORREKT: auto += 2
            except Exception: pass
            # 5.1 a) Reihenfolge (4 Pkt.) + c) Frist (3 Pkt.)
            try:
                if (schritt_1.value == pruef_schritte[0] and
                        schritt_2.value == pruef_schritte[1] and
                        schritt_3.value == pruef_schritte[2]):
                    auto += 4
            except Exception: pass
            try:
                if h(pruef_frist.value or '') == _H_1J: auto += 3
            except Exception: pass
            # ─ Manuell ─
            manuell = 0.0
            for aufg, (sl, _) in bewertungs_inputs.items():
                manuell += sl.value * 0.5 if aufg == '1.1b' else float(sl.value)
            gesamt = round(auto + bonus + manuell, 1)
            if   gesamt >= 92: note = '1 – Sehr gut'
            elif gesamt >= 81: note = '2 – Gut'
            elif gesamt >= 67: note = '3 – Befriedigend'
            elif gesamt >= 50: note = '4 – Ausreichend'
            elif gesamt >= 20: note = '5 – Nicht bestanden'
            else:              note = '6 – Ungenügend'
            bonus_str = (f" + <b style='color:#0077b6;'>{bonus:.0f} Bonus (W3)</b>"
                         if bonus else '')
            with out_gesamt:
                out_gesamt.clear_output()
                display(HTML(
                    f"<div style='background:#d4edda;border-left:4px solid #28a745;"
                    f"padding:12px;border-radius:6px;margin-top:8px;'>"
                    f"<b>Auswertung für {p_name}</b><br>"
                    f"Automatisch: <b>{auto:.1f} / 61,5 Pkt.</b>{bonus_str}<br>"
                    f"Manuell: <b>{manuell:.1f} / 38,5 Pkt.</b><br>"
                    f"<b>Gesamt: {gesamt:.1f} / 106 Pkt.</b><br><br>"
                    f"<b style='font-size:1.2em;'>Note: {note}</b>"
                    f"</div>"
                ))

        btn_berechne.on_click(berechne_gesamt)
        display(widgets.VBox([
            widgets.HBox([pwd_w, btn_unlock]), out_unlock,
            *rows, btn_berechne, out_gesamt
        ]))

out_abgabe = widgets.Output()

def abgabe(b):
    b.disabled=True; b.description='✅ Abgegeben'; b.button_style='success'
    log("Klausur abgegeben", f"Versuchsprotokoll: {_versuche}")
    sperre_alle()
    zeige_feedback()
    try: zeige_wt_21()
    except Exception: pass
    with out_abgabe:
        out_abgabe.clear_output(wait=True)
        display(HTML(
            "<div style='background:#fff3cd;border-left:4px solid #ffc107;"
            "padding:12px;margin-top:12px;border-radius:4px;'>"
            "<b>Hinweis:</b> Feedback mit Leitstand-Beobachtungen vergleichen."
            "</div>"
        ))
        erstelle_bewertungsfelder()
        display(out_bewertung)
        try:
            pfad = exportiere_abgabe()
            display(HTML(
                f"<div style='background:#d4edda;border-left:4px solid #28a745;"
                f"padding:10px;border-radius:4px;margin-top:10px;font-size:.9em;'>"
                f"Abgabe gespeichert: <code>{pfad}</code></div>"
            ))
        except Exception as e:
            display(HTML(
                f"<div style='background:#f8d7da;border-left:4px solid #dc3545;"
                f"padding:10px;border-radius:4px;margin-top:10px;font-size:.9em;'>"
                f"Export fehlgeschlagen: {e}</div>"
            ))
    print(f"Prüfung abgegeben: {p_name} | {p_klass}")
    print(f"Versuchsprotokoll: {_versuche}")

btn_abg = widgets.Button(
    description='Klausur final abgeben',
    button_style='danger',
    icon='check-circle',
    layout={'width': '300px', 'height': '50px'}
)
btn_abg.on_click(abgabe)

display(widgets.VBox([
    widgets.HTML('<br><hr><b>Abgabe</b>'),
    widgets.Label(
        'Nach Klick: Alle Widgets gesperrt, Feedback eingeblendet, '
        'Bewertungsfelder für Lehrkraft sichtbar.'
    ),
    btn_abg,
    out_abgabe
]))
